# Supplementary Figure 10 | AlphaGenome prediction of fine-mapped eQTL variants.

**a.** High confidence QTL training set size (variant-gene-tissue sets) for GTEx v10 versus v11. Bars are colored by biotype of the QTL gene, protein-coding (p.c.), existing lncRNA, or novel lncRNA. v11 adds novel lncRNA QTLs and expands existing lncRNA QTLs relative to v10.

**b.** Tissue-weighted auROC versus training set size for random-forest (RF) models trained on v11 QTL targets from one biotype (color) and evaluated on held-out chromosome-fold QTL targets of existing lncRNA, novel lncRNA, or pooled protein-coding + lncRNA (protein-coding holdout learning curve in Figure 3f). Points, mean auROC ± fold-level 95% CI across five chromosome folds. Models trained on lncRNA QTLs transfer to held-out QTLs at performance comparable to protein-coding-trained models across biotypes, consistent with shared sequence features of fine-mapped eQTL variants across biotypes.

**c.** Tissue-weighted auROC for zero-shot AlphaGenome scoring, v10-supervised RF, and v11-supervised RF on held-out chromosome-fold cohorts of v10 protein-coding, v10 lncRNA, v11 protein-coding, and v11 lncRNA QTL targets. Bars, mean auROC ± fold-level 95% CI across five chromosome folds; asterisks, two-sided paired t-test across folds (*, p < 0.05; **, p < 0.01; ***, p < 0.001; ****, p < 0.0001). Supervised models improve over zero-shot, but v11 does not significantly improve over v10, likely due to saturation of useful supervision at v10 label scales.


## Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.gridspec import GridSpec

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

pip_cutoff = 0.9
score_biotypes = ["lncRNA", "protein_coding"]
count_biotypes = ["existing_lncRNA", "novel_lncRNA", "protein_coding"]
label_subsample_grid = [50, 100, 500, 1000, 5000, 10000, 50000]
label_subsample_train_biotypes = ["protein_coding", "novel_lncRNA", "existing_lncRNA"]
# Panel b eval holdouts (protein-coding holdout learning curve is Figure 3f).
curve_evals = ["existing_lncRNA", "novel_lncRNA", "all_genes"]
panel_c_score_cols = ["zero_shot_abs", "rf_v10", "rf_v11"]
panel_c_keys = ["v10_protein_coding", "v10_lncRNA", "v11_protein_coding", "v11_lncRNA"]

paired = sns.color_palette("Paired", 12)
biotype_palette = {
    "lncRNA": paired[5],
    "novel_lncRNA": paired[5],
    "existing_lncRNA": paired[4],
    "protein_coding": paired[1],
}
biotype_pretty = {
    "all_genes": "p.c. + lncRNA",
    "lncRNA": "lncRNA",
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
}
panel_c_pretty = {
    "v10_protein_coding": "v10 protein-coding",
    "v10_lncRNA": "v10 lncRNA",
    "v11_protein_coding": "v11 protein-coding",
    "v11_lncRNA": "v11 lncRNA",
}
score_xticklabels = ["Zero-shot", "v10 RF", "v11 RF"]
score_bar_colors = ["#d9d9d9", "#969696", "#525252"]


## Paths

In [ ]:
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")
v10_rf_examples_file = Path("path/to/v10_qtl_variant_features.parquet")
v11_rf_examples_file = Path("path/to/v11_qtl_variant_features.parquet")
learning_curves_file = Path("path/to/qtl_causality_auroc.tsv")

fig_dir = Path("figures")


## Load data


### Supervision cohort sizes (panel a)


In [ ]:
# Panel a: all available PIP≥0.9 positive labels (pre distance-matching), not the 50/50 RF cohort.

def _is_snv_variant_id(variant_id: str) -> bool:
    # GTEx IDs are chr_pos_ref_alt[_b38]; SNV iff ref/alt are single ACGT bases.
    parts = str(variant_id).split("_")
    if len(parts) < 4:
        return False
    ref, alt = parts[2], parts[3]
    return len(ref) == 1 and len(alt) == 1 and ref in "ACGT" and alt in "ACGT"

_lnc_split = {"novel_lncRNA", "existing_lncRNA"}
gene_class = pd.read_csv(
    gencode_class_file,
    sep="\t",
    usecols=["gene_id_stable", "gene_type_grouped"],
    dtype=str,
).dropna(subset=["gene_id_stable", "gene_type_grouped"])
split_lookup = gene_class.set_index("gene_id_stable")["gene_type_grouped"].to_dict()

rf_example_paths = {"v10": v10_rf_examples_file, "v11": v11_rf_examples_file}
cohort_rows = []
for version, path in rf_example_paths.items():
    pos = pd.read_parquet(path, columns=["variant_id", "pip", "gene_id_stable"])
    pos = pos[pos["pip"] >= pip_cutoff].copy()
    pos = pos[pos["variant_id"].map(_is_snv_variant_id)].copy()
    pos["split_biotype"] = pos["gene_id_stable"].map(split_lookup)
    pos["gene_biotype"] = np.where(
        pos["split_biotype"].isin(list(_lnc_split)), "lncRNA", pos["split_biotype"]
    )
    pos = pos[pos["gene_biotype"].isin(score_biotypes)].copy()
    if version == "v10":
        pos.loc[pos["gene_biotype"] == "lncRNA", "split_biotype"] = "existing_lncRNA"
    for biotype, sub in pos.groupby("split_biotype"):
        if biotype not in count_biotypes:
            continue
        cohort_rows.append(
            {
                "version": version,
                "gene_biotype": biotype,
                "n_examples": int(len(sub)),
            }
        )

cohort_sizes = pd.DataFrame(cohort_rows)
print(f"cohort_sizes: {len(cohort_sizes)} rows")
print(cohort_sizes.pivot(index="gene_biotype", columns="version", values="n_examples").reindex(count_biotypes))
print("totals", cohort_sizes.groupby("version")["n_examples"].sum().to_dict())


### Label-subsample curves (panel b)


In [ ]:
learning_curves = pd.read_csv(learning_curves_file, sep="\t")
# Matched-N / Fig 3f-style rows are v11 subsample curves (not catalog holdouts).
is_catalog = (
    learning_curves["train_biotype"].isin(["zero_shot", "rf_v10", "rf_v11"])
    & learning_curves["eval_biotype"].isin(["lncRNA", "protein_coding"])
)
curve_rows = learning_curves[
    ~is_catalog
    & (
        learning_curves["dataset_version"].isna()
        | (learning_curves["dataset_version"].astype(str) == "v11")
    )
].copy()
print(
    f"Loaded {learning_curves_file}; "
    f"{len(learning_curves)} rows "
    f"({len(curve_rows)} matched-N/zero-shot curve rows); "
    f"evals={sorted(curve_rows['eval_biotype'].unique())}"
)


### Catalog comparison auROCs (panel c)

In [ ]:
if "dataset_version" not in learning_curves.columns:
    raise ValueError(
        f"{learning_curves_file.name} is missing catalog-comparison rows "
        "(no dataset_version column). Refresh with alphagenome_qtl/04_train_subsample_rf.py "
        "--catalog-metrics-only."
    )

catalog_curves = learning_curves[
    learning_curves["train_biotype"].isin(["zero_shot", "rf_v10", "rf_v11"])
    & learning_curves["eval_biotype"].isin(["lncRNA", "protein_coding"])
    & learning_curves["dataset_version"].isin(["v10", "v11"])
].copy()
catalog_curves["condition"] = catalog_curves["train_biotype"].replace(
    {"zero_shot": "zero_shot_abs"}
)
catalog_curves["holdout"] = (
    catalog_curves["dataset_version"].astype(str)
    + "_"
    + catalog_curves["eval_biotype"].astype(str)
)
missing = [k for k in panel_c_keys if k not in set(catalog_curves["holdout"])]
if missing:
    raise FileNotFoundError(
        f"Missing catalog holdouts in {learning_curves_file.name}: {missing}"
    )
# Expect one fold-aggregated row per holdout × method (3 methods × 4 holdouts).
counts = catalog_curves.groupby("holdout").size()
bad = [k for k in panel_c_keys if int(counts.get(k, 0)) != len(panel_c_score_cols)]
if bad:
    raise ValueError(
        f"Expected {len(panel_c_score_cols)} aggregated catalog rows per holdout; "
        f"bad={ {k: int(counts.get(k, 0)) for k in bad} }. "
        "Refresh with alphagenome_qtl/04_train_subsample_rf.py --catalog-metrics-only."
    )
print(
    f"Catalog fold-aggregated auROCs from {learning_curves_file.name}: "
    f"{len(catalog_curves)} rows; "
    + ", ".join(
        f"{k}={int((catalog_curves['holdout'] == k).sum())}" for k in panel_c_keys
    )
)

## Run analysis


### Panel c aggregated auROCs and paired tests


In [ ]:
def sig_stars(p):
    if not np.isfinite(p):
        return "ns"
    if p < 0.0001:
        return "****"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def catalog_bar_table(holdout_key):
    """Read fold-aggregated catalog mean±CI rows for one holdout."""
    sub = catalog_curves[catalog_curves["holdout"] == holdout_key].copy()
    sub = sub.set_index("condition").reindex(panel_c_score_cols)
    if sub["mean"].isna().any():
        missing = [c for c in panel_c_score_cols if pd.isna(sub.loc[c, "mean"])]
        raise ValueError(
            f"Incomplete catalog auROCs for {holdout_key}: missing={missing}"
        )
    rows = []
    for i, col in enumerate(panel_c_score_cols):
        rows.append(
            {
                "condition": col,
                "i": i,
                "mean": float(sub.loc[col, "mean"]),
                "ci_lower": float(sub.loc[col, "ci_lower"]),
                "ci_upper": float(sub.loc[col, "ci_upper"]),
                "n_folds": int(sub.loc[col, "n_folds"]),
            }
        )
    return pd.DataFrame(rows)


def catalog_paired_tests(holdout_key):
    """Rebuild pairwise bracket table from precomputed fold-paired p-values."""
    sub = catalog_curves[catalog_curves["holdout"] == holdout_key].set_index("condition")
    means = sub["mean"].astype(float)
    # score cols: zero_shot_abs, rf_v10, rf_v11
    pairs = [
        ("zero_shot_abs", "rf_v10", "p_vs_zero_shot"),
        ("zero_shot_abs", "rf_v11", "p_vs_zero_shot"),
        ("rf_v10", "rf_v11", "p_vs_rf_v10"),
    ]
    rows = []
    for a, b, pcol in pairs:
        if pcol not in sub.columns:
            raise ValueError(
                f"{learning_curves_file.name} missing {pcol}; refresh catalog metrics."
            )
        p = float(sub.loc[b, pcol])
        rows.append(
            {
                "i": panel_c_score_cols.index(a),
                "j": panel_c_score_cols.index(b),
                "a": a,
                "b": b,
                "comparison": f"{b}  vs  {a}",
                "n_folds": int(sub.loc[b, "n_folds"]),
                "mean_delta_(b-a)": float(means[b] - means[a]),
                "p_paired_t": p,
                "stars": sig_stars(p),
            }
        )
    return pd.DataFrame(rows)


panel_bar = {}
pval_rows = []
for key in panel_c_keys:
    panel_bar[key] = catalog_bar_table(key)
    pw = catalog_paired_tests(key)
    pw["biotype"] = key
    pval_rows.append(pw)

pval_table = pd.concat(pval_rows, ignore_index=True)
print(f"panel c auROC summaries: {len(panel_c_keys)} holdouts ({', '.join(panel_c_keys)})")


## Make figure


In [ ]:
def add_sig_brackets(ax, pairs, y_max, y_min, *, fontsize=6):
    span = y_max - y_min
    top = y_max
    for k, row in enumerate(pairs):
        x1, x2 = row["i"], row["j"]
        stars = row["stars"]
        if stars == "ns":
            continue
        y = top + span * (0.04 + 0.08 * k)
        ax.plot([x1, x1, x2, x2], [y, y + span * 0.015, y + span * 0.015, y], color="0.2", linewidth=0.6, clip_on=False)
        ax.text(
            0.5 * (x1 + x2),
            y + span * 0.02,
            stars,
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="0.15",
            clip_on=False,
        )
        top = max(top, y + span * 0.08)
    return top


fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
gs = GridSpec(
    2,
    4,
    figure=fig,
    height_ratios=[1.1, 1.0],
    hspace=0.4,
    wspace=0.75,
)
ax_a = fig.add_subplot(gs[0, 0])
axes_sub = [fig.add_subplot(gs[0, i]) for i in range(1, 4)]
axes_tissue = [fig.add_subplot(gs[1, i]) for i in range(4)]

# Panel a — stacked cohort sizes (compact labels)
plot_sizes = cohort_sizes[cohort_sizes["gene_biotype"].isin(count_biotypes)].copy()
stack_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]
short_pretty = {
    "novel_lncRNA": "novel",
    "existing_lncRNA": "existing",
    "protein_coding": "p.c.",
}
plot_sizes["label"] = pd.Categorical(
    plot_sizes["gene_biotype"].map(short_pretty),
    categories=[short_pretty[b] for b in stack_order],
    ordered=True,
)
plot_sizes["version"] = pd.Categorical(plot_sizes["version"], categories=["v10", "v11"], ordered=True)
sns.histplot(
    data=plot_sizes,
    x="version",
    weights="n_examples",
    hue="label",
    multiple="stack",
    palette=[biotype_palette[b] for b in stack_order],
    shrink=0.7,
    edgecolor="white",
    linewidth=0.5,
    discrete=True,
    ax=ax_a,
    legend=True,
    hue_order=[short_pretty[b] for b in stack_order],
)
sns.move_legend(
    ax_a,
    "lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=False,
    title="",
    columnspacing=0.6,
    handletextpad=0.3,
    borderpad=0.1,
    handlelength=1.0,
    labelspacing=0.2,
)
if ax_a.legend_ is not None:
    for text in ax_a.legend_.get_texts():
        text.set_fontsize(3.8)
for c in ax_a.containers:
    labels = [f"{v/1000:.0f}k" if v >= 1000 else (f"{v:.0f}" if v > 0 else "") for v in c.datavalues]
    ax_a.bar_label(c, labels=labels, label_type="center", color="black", fontsize=3.8)
totals = plot_sizes.groupby("version", observed=True)["n_examples"].sum().reindex(["v10", "v11"])
for i, (version, total) in enumerate(totals.items()):
    if pd.isna(total):
        continue
    total = float(total)
    ax_a.text(
        i,
        total,
        f"{total/1000:.0f}k",
        ha="center",
        va="bottom",
        fontsize=4.2,
        fontweight="medium",
        clip_on=False,
    )
ax_a.set_xticks(np.arange(2))
ax_a.set_xticklabels(["v10", "v11"])
ax_a.set_xlabel("GTEx version")
ax_a.set_ylabel("variant–gene–tissue QTLs w/ PIP ≥ 0.9")
ax_a.set_ylim(0, 110_000)
ax_a.set_yticks([0, 50000, 100000])
ax_a.set_yticklabels(["0", "50k", "100k"])
sns.despine(ax=ax_a)

# Panel b — label-subsample curves
for ax, eval_bt in zip(axes_sub, curve_evals):
    eval_curves = curve_rows[curve_rows["eval_biotype"] == eval_bt]
    x_max = label_subsample_grid[-1]
    for train_bt in label_subsample_train_biotypes:
        curve = (
            eval_curves[eval_curves["train_biotype"] == train_bt]
            .sort_values("n_train")
            .copy()
        )
        if curve.empty:
            continue
        color = biotype_palette.get(train_bt, "#333333")
        yerr_lo = curve["mean"] - curve["ci_lower"]
        yerr_hi = curve["ci_upper"] - curve["mean"]
        ax.errorbar(
            curve["n_train"],
            curve["mean"],
            yerr=[yerr_lo, yerr_hi],
            fmt="-o",
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=0,
            markersize=3.0,
            markeredgecolor="k",
            markeredgewidth=0.35,
            linewidth=0.8,
            zorder=3,
            label=biotype_pretty[train_bt],
        )
        x_max = max(x_max, int(curve["n_train"].max()))
    ax.set_xscale("log")
    ax.set_xlim(label_subsample_grid[0] * 0.8, x_max * 1.15)
    ax.set_xticks([100, 1000, 10000, 50000])
    ax.set_xticklabels([100, "1k", "10k", "50k"])
    ax.set_xlabel("RF training set size")
    ax.set_ylabel(f"tissue-weighted auROC\non {biotype_pretty[eval_bt]}")
    ax.legend(frameon=False, loc="lower right", fontsize=4.5, title="Training biotype")
    ax.set_ylim(0.5, 0.8)
    sns.despine(ax=ax)

# Panel c — zero-shot / v10 RF / v11 RF bars
xs = np.arange(len(panel_c_score_cols))
bar_ymax = max(float(panel_bar[b]["ci_upper"].max()) for b in panel_c_keys)
for ax, biotype in zip(axes_tissue, panel_c_keys):
    bar = panel_bar[biotype]
    means = bar["mean"].to_numpy(dtype=float)
    yerr = np.vstack([means - bar["ci_lower"].to_numpy(), bar["ci_upper"].to_numpy() - means])
    ax.bar(
        xs,
        means,
        width=0.72,
        color=score_bar_colors,
        edgecolor="k",
        linewidth=0.4,
        zorder=2,
    )
    ax.errorbar(
        xs,
        means,
        yerr=yerr,
        fmt="none",
        ecolor="0.2",
        elinewidth=0.8,
        capsize=0,
        zorder=3,
    )
    pw = pval_table[pval_table["biotype"] == biotype].to_dict("records")
    top = add_sig_brackets(ax, pw, bar_ymax, 0.5, fontsize=5.5)
    ax.set_xticks(xs)
    ax.set_xticklabels(score_xticklabels, fontsize=4.5)
    ax.set_ylabel(f"tissue-weighted auROC\non {panel_c_pretty[biotype]}")
    ax.set_xlim(-0.55, len(panel_c_score_cols) - 0.45)
    ax.set_ylim(0.5, top)
    sns.despine(ax=ax)

panel_label_kwargs = {
    "fontweight": "bold",
    "va": "bottom",
    "ha": "right",
    "fontsize": 9,
    "clip_on": False,
}

pos_a = ax_a.get_position()
pos_b = axes_sub[0].get_position()
pos_c = axes_tissue[0].get_position()


def _axes_to_fig(ax, x, y):
    return fig.transFigure.inverted().transform(ax.transAxes.transform((x, y)))


y_ab = max(_axes_to_fig(ax_a, 0, 1.08)[1], _axes_to_fig(axes_sub[0], 0, 1.08)[1])
y_c = _axes_to_fig(axes_tissue[0], 0, 1.08)[1]
x_ac = min(pos_a.x0, pos_c.x0) - 0.06
x_b = pos_b.x0 - 0.06
fig.text(x_ac, y_ab, "a", transform=fig.transFigure, **panel_label_kwargs)
fig.text(x_b, y_ab, "b", transform=fig.transFigure, **panel_label_kwargs)
fig.text(x_ac, y_c, "c", transform=fig.transFigure, **panel_label_kwargs)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_10_alphagenome.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_10_alphagenome.png", bbox_inches="tight")
print("Wrote", fig_dir / "supp_figure_10_alphagenome.pdf")
print("Wrote", fig_dir / "supp_figure_10_alphagenome.png")

## Print stats


In [ ]:
# Caption / Results stats for Supplementary Figure 10.
n_total = cohort_sizes.groupby("version")["n_examples"].sum()
n_lnc = (
    cohort_sizes[cohort_sizes["gene_biotype"].isin(["existing_lncRNA", "novel_lncRNA"])]
    .groupby("version")["n_examples"]
    .sum()
)
n_pc = cohort_sizes[cohort_sizes["gene_biotype"] == "protein_coding"].groupby("version")["n_examples"].sum()
print(
    f"eQTL labels v10 → v11: {n_total['v10']:,} → {n_total['v11']:,} "
    f"({100 * (n_total['v11'] / n_total['v10'] - 1):+.0f}%)"
)
print(f"lncRNA: {n_lnc['v10']:,} → {n_lnc['v11']:,}")
print(f"protein-coding: {n_pc['v10']:,} → {n_pc['v11']:,}")

delta_vs_zero_shot = []
for key in ["v10_protein_coding", "v10_lncRNA"]:
    bar = panel_bar[key].set_index("condition")["mean"]
    delta_vs_zero_shot.append(bar["rf_v11"] - bar["zero_shot_abs"])
print(f"tissue-weighted ΔauROC, supervised vs zero-shot = {np.mean(delta_vs_zero_shot):+.3f}")

# Results: expanding v10 → v11 labels yields little further gain on the shared v10 evaluation
delta_v11_vs_v10 = []
for key in ["v10_protein_coding", "v10_lncRNA"]:
    bar = panel_bar[key].set_index("condition")["mean"]
    delta_v11_vs_v10.append(bar["rf_v11"] - bar["rf_v10"])
print(f"tissue-weighted ΔauROC, v11- vs v10-supervised on v10 eval = {np.mean(delta_v11_vs_v10):+.3f}")
